# CSP — atribuição de encomendas a veículos

Este notebook apresenta uma primeira formulação do problema de satisfação de restrições (CSP) para o projeto. O objetivo é atribuir cada encomenda completa a um veículo, comparar o solver `python-constraint` com um backtracking próprio e observar o efeito do MRV.

O enunciado pede que se apresentem variáveis, domínios e restrições, que se expliquem as heurísticas usadas nas restrições suaves e que se reportem uma atribuição viável e o número de backtracks.

## 1. Decisões de modelação

- Uma variável representa uma encomenda (`order_id`), não uma linha de produto. Assim, todos os itens da encomenda ficam com o mesmo veículo.
- O domínio de cada variável contém os veículos que podem transportar a encomenda e para os quais a encomenda é temporalmente viável quando planeada sozinha.
- `single_load` aplica o limite de capacidade à soma das encomendas atribuídas a cada veículo.
- `multi_trip` verifica a capacidade por encomenda, permitindo que o veículo faça várias viagens. Este modo corresponde melhor ao modelo de rotas em `model.py`.
- A heurística suave prefere cargas normalizadas equilibradas. É usada para ordenar escolhas, sem proibir uma solução válida por ser desequilibrada.

A verificação temporal isolada não garante que várias encomendas atribuídas ao mesmo veículo possam ser encadeadas numa rota válida. A rota final deve continuar a ser validada pelo planeador/verificador.

In [1]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'model.py').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from model import Problem
from csp.csp import FleetCSP, compare_backtracking

problem = Problem(ROOT / 'docs' / 'data_tiny')
csp = FleetCSP(problem)
print(f'Encomendas: {len(csp.tasks)} | Veículos: {csp.vehicle_ids}')
print(f'Início temporal usado: {csp.planning_start}')

Encomendas: 12 | Veículos: (1, 2)
Início temporal usado: 2026-10-30 00:00:00


## 2. Variáveis e domínios

Cada linha da tabela abaixo representa uma variável. `eligible_vehicles` mostra o seu domínio depois das restrições unárias de capacidade, janelas e compatibilidade.

In [2]:
domains = pd.DataFrame([
    {
        'Variável': f'order_{task.order_id}',
        'Itens': list(task.item_ids),
        'Peso (kg)': round(task.weight_kg, 2),
        'Domínio': list(csp.domains[task.order_id]),
    }
    for task in csp.tasks
])
domains

,Variável,Itens,Peso (kg),Domínio
0,order_1,"[1, 2, 3, 4]",22.0,"[1, 2]"
1,order_2,"[5, 6]",6.8,"[1, 2]"
2,order_3,"[7, 8, 9, 10]",21.6,"[1, 2]"
3,order_4,"[11, 12, 13]",26.3,"[1, 2]"
4,order_5,"[14, 15]",9.3,"[1, 2]"
5,order_6,"[16, 17]",16.7,"[1, 2]"
6,order_7,"[18, 19, 20, 21]",33.5,"[1, 2]"
7,order_8,"[22, 23, 24]",18.8,"[1, 2]"
8,order_9,"[25, 26, 27, 28, 29]",36.7,"[1, 2]"
9,order_10,"[30, 31]",9.2,"[1, 2]"


### Restrições consideradas

1. **Encomenda indivisível:** é satisfeita por haver uma só variável por encomenda.
2. **Capacidade:** cada encomenda tem de caber no veículo; em `single_load`, a soma atribuída também respeita a capacidade total.
3. **Janelas temporais:** a encomenda tem de admitir alguma ordem de visita aos produtores, passagem pelo armazém e entrega dentro das janelas.
4. **Compatibilidade:** produtos que indiquem `requires_refrigeration=True` só podem ser atribuídos a veículos refrigerados. O dataset atual não inclui esse atributo, pelo que não se infere refrigeração pela categoria do produto.

A restrição suave é o equilíbrio de carga: a ordenação de valores tenta minimizar `max(peso/capacidade) - min(peso/capacidade)`.

## 3. Resolver com `python-constraint`

A enumeração de soluções está limitada a 5 000 para o notebook permanecer interativo. Se o solver esgotar todas as soluções, `optimal_balance_proven` indica que a solução escolhida tem o melhor equilíbrio entre as soluções viáveis enumeradas.

In [3]:
constraint_result = csp.solve_python_constraint(
    capacity_mode='single_load',
    max_solutions=5_000,
)
constraint_result

CSPResult(method='python_constraint', feasible=True, assignment={12: 2, 9: 1, 7: 1, 4: 2, 1: 1, 3: 2, 8: 2, 11: 2, 6: 1, 5: 2, 10: 2, 2: 1}, loads_kg={1: 115.7, 2: 154.3}, balance_score=0.00016666666666675933, backtracks=0, nodes=0, solutions_examined=2257, optimal_balance_proven=True, capacity_mode='single_load', message='A melhor solução equilibrada foi encontrada em todas as soluções.')

In [4]:
assignment_table = pd.DataFrame(csp.summary_rows(constraint_result))
assignment_table

,order_id,vehicle_id,item_ids,weight_kg,eligible_vehicles
0,1,1,"[1, 2, 3, 4]",22.0,"[1, 2]"
1,2,1,"[5, 6]",6.8,"[1, 2]"
2,3,2,"[7, 8, 9, 10]",21.6,"[1, 2]"
3,4,2,"[11, 12, 13]",26.3,"[1, 2]"
4,5,2,"[14, 15]",9.3,"[1, 2]"
5,6,1,"[16, 17]",16.7,"[1, 2]"
6,7,1,"[18, 19, 20, 21]",33.5,"[1, 2]"
7,8,2,"[22, 23, 24]",18.8,"[1, 2]"
8,9,1,"[25, 26, 27, 28, 29]",36.7,"[1, 2]"
9,10,2,"[30, 31]",9.2,"[1, 2]"


### Interpretar a solução

- `assignment` associa cada `order_id` a um `vehicle_id`.
- `loads_kg` soma o peso das encomendas por veículo.
- `balance_score` mede a diferença entre as cargas normalizadas dos veículos; valores menores indicam maior equilíbrio.
- O score é uma preferência, não uma restrição rígida.

Se não houver solução em `single_load`, experimente `multi_trip`. Nesse modo, cada encomenda continua a respeitar a capacidade do veículo, mas a soma das encomendas pode ultrapassar essa capacidade porque o veículo pode regressar ao armazém.

In [5]:
multi_trip_result = csp.solve_backtracking(
    use_mrv=True,
    capacity_mode='multi_trip',
)
pd.DataFrame(csp.summary_rows(multi_trip_result)), multi_trip_result.loads_kg

(    order_id  vehicle_id              item_ids  weight_kg eligible_vehicles
 0          1           2          [1, 2, 3, 4]       22.0            [1, 2]
 1          2           1                [5, 6]        6.8            [1, 2]
 2          3           1         [7, 8, 9, 10]       21.6            [1, 2]
 3          4           1          [11, 12, 13]       26.3            [1, 2]
 4          5           1              [14, 15]        9.3            [1, 2]
 5          6           2              [16, 17]       16.7            [1, 2]
 6          7           2      [18, 19, 20, 21]       33.5            [1, 2]
 7          8           2          [22, 23, 24]       18.8            [1, 2]
 8          9           1  [25, 26, 27, 28, 29]       36.7            [1, 2]
 9         10           2              [30, 31]        9.2            [1, 2]
 10        11           1      [32, 33, 34, 35]       18.0            [1, 2]
 11        12           2  [36, 37, 38, 39, 40]       51.1            [1, 2]

## 4. Backtracking próprio: sem MRV e com MRV

O contador de backtracks aumenta quando uma escolha é abandonada por não permitir completar a atribuição. `nodes` conta estados visitados pela recursão. O MRV escolhe, a cada passo, a encomenda com menos veículos ainda possíveis, desempata por peso, prazo e ID. A ordenação dos veículos usa a heurística de equilíbrio.

In [6]:
backtracking_results = compare_backtracking(csp, capacity_mode='single_load')
comparison = pd.DataFrame([
    {
        'Método': r.method,
        'Viável': r.feasible,
        'Backtracks': r.backtracks,
        'Nós': r.nodes,
        'Equilíbrio': r.balance_score,
        'Carga por veículo (kg)': r.loads_kg,
    }
    for r in backtracking_results
])
comparison

,Método,Viável,Backtracks,Nós,Equilíbrio,Carga por veículo (kg)
0,backtracking_sem_mrv,True,0,13,0.202000,"{1: 98.4, 2: 171.6}"
1,backtracking_mrv,True,0,13,0.034833,"{1: 118.69999999999999, 2: 151.29999999999998}"


## 5. O que analisar e completar

1. Registe a atribuição viável, a carga por veículo, os nós e os backtracks.
2. Compare o backtracking estático com MRV. Explique se a ordem das variáveis alterou o esforço nesta instância.
3. Compare a primeira solução do solver com a solução selecionada pelo balanceamento.
4. Repita no conjunto `small` com `capacity_mode='multi_trip'`; explique porque o modo `single_load` pode ser inviável quando o peso total ultrapassa a capacidade agregada da frota.
5. Valide a atribuição escolhida com o planeador de rotas. A CSP atual não resolve conflitos temporais entre encomendas diferentes no mesmo veículo.

As células não têm resultados pré-calculados: executar o notebook mostra os resultados produzidos no ambiente do grupo.